## Silver → Gold: Business Aggregations

Joins cleaned trip data with the zone lookup dimension table and builds four business-ready aggregated tables answering key demand/revenue questions.

**Flow:**
1. Read silver Delta table and zone lookup reference data.
2. Join trip data to zone lookup twice (pickup and dropoff), using a left join to avoid silently dropping trips with unmatched location IDs.
3. Build four gold tables: monthly trend, revenue by borough/hour, tips by payment type, top pickup zones.
4. Write each as its own Delta table in gold.

### Step 1: Join with Zone Lookup (Pickup and Dropoff)

Joins twice against the same zone lookup table — once for pickup location, once for dropoff — using a left join so trips with an unmatched/invalid zone ID aren't silently dropped.

In [0]:
storage_account = "stnyctaxiraj01"

df_silver = spark.read.format("delta").load(f"abfss://silver@{storage_account}.dfs.core.windows.net/taxi/")
df_zones = spark.read.option("header", "true").option("inferSchema", "true").csv(
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/zones/taxi_zone_lookup.csv"
)

In [0]:
from pyspark.sql.functions import col

# Alias the zone lookup table twice with distinct prefixes
df_zones_pu = df_zones.select(
    col("LocationID").alias("PULocationID"),
    col("Borough").alias("PU_Borough"),
    col("Zone").alias("PU_Zone")
)

df_zones_do = df_zones.select(
    col("LocationID").alias("DOLocationID"),
    col("Borough").alias("DO_Borough"),
    col("Zone").alias("DO_Zone")
)

df_joined = df_silver \
    .join(df_zones_pu, on="PULocationID", how="left") \
    .join(df_zones_do, on="DOLocationID", how="left")

print(f"Joined row count: {df_joined.count()}")
df_joined.select("PULocationID", "PU_Borough", "PU_Zone", "DOLocationID", "DO_Borough", "DO_Zone").show(5)

### Step 2: Build the Four Gold Aggregation Tables

1. **Monthly trend** — trip count, revenue, avg fare by month
2. **Revenue by borough/hour** — demand patterns by time of day and location
3. **Tips by payment type** — average tip % (not raw amount, for comparability)
4. **Top pickup zones** — busiest 20 zones by trip volume

In [0]:
from pyspark.sql.functions import hour, count, sum as _sum, avg, round as _round

df_gold_borough_hour = df_joined \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .groupBy("PU_Borough", "pickup_hour") \
    .agg(
        count("*").alias("trip_count"),
        _round(_sum("total_amount"), 2).alias("total_revenue"),
        _round(avg("total_amount"), 2).alias("avg_fare")
    ) \
    .orderBy("PU_Borough", "pickup_hour")

df_gold_borough_hour.show(10)

In [0]:
from pyspark.sql.functions import when

df_gold_tips = df_joined \
    .withColumn("tip_pct", _round((col("tip_amount") / col("fare_amount")) * 100, 2)) \
    .groupBy("payment_type") \
    .agg(
        count("*").alias("trip_count"),
        _round(avg("tip_pct"), 2).alias("avg_tip_pct")
    ) \
    .orderBy("payment_type")

df_gold_tips.show()

In [0]:
df_gold_top_zones = df_joined \
    .groupBy("PU_Borough", "PU_Zone") \
    .agg(
        count("*").alias("trip_count"),
        _round(_sum("total_amount"), 2).alias("total_revenue")
    ) \
    .orderBy(col("trip_count").desc()) \
    .limit(20)

df_gold_top_zones.show(20, truncate=False)

In [0]:
df_gold_monthly_trend = df_joined \
    .groupBy("year", "month") \
    .agg(
        count("*").alias("trip_count"),
        _round(_sum("total_amount"), 2).alias("total_revenue"),
        _round(avg("total_amount"), 2).alias("avg_fare")
    ) \
    .orderBy("year", "month")

df_gold_monthly_trend.show(12)

%md
### Step 3: Write All Four Tables to Gold as Delta

In [0]:
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/"

df_gold_borough_hour.write.format("delta").mode("overwrite").save(f"{gold_path}revenue_by_borough_hour/")
df_gold_tips.write.format("delta").mode("overwrite").save(f"{gold_path}tips_by_payment_type/")
df_gold_top_zones.write.format("delta").mode("overwrite").save(f"{gold_path}top_pickup_zones/")
df_gold_monthly_trend.write.format("delta").mode("overwrite").save(f"{gold_path}monthly_trend/")

print("All gold tables written.")

In [0]:
for name in ["revenue_by_borough_hour", "tips_by_payment_type", "top_pickup_zones", "monthly_trend"]:
    df_check = spark.read.format("delta").load(f"{gold_path}{name}/")
    print(f"{name}: {df_check.count()} rows")